# Giorno 3 — Image/Video Restoration & Caso Studio: Targa

**Obiettivi della giornata:**
1. Restoration di frame CCTV con Real-ESRGAN (super-resolution, denoising)
2. Face restoration con GFPGAN su volti degradati
3. Ripetibilità: logging, salvataggio parametri, hash delle immagini
4. **Caso studio completo**: pipeline di restoration di targhe degradate
   - Detection targa (YOLOv11 pre-addestrato su targhe)
   - Restoration (Real-ESRGAN)
   - Confronto visivo originale vs restaurato

---
**Dataset:**
- Video CCTV a bassa risoluzione (`CCTV Action Recognition`)
- Immagini di volti da sorveglianza (`SCface`)
- Immagini di targhe personalizzate

**Modelli:** Real-ESRGAN (restoration generale), GFPGAN (face restoration), YOLOv11 (license plate)

> Anche oggi: solo inference, nessun retraining.

## 0. Setup

In [ ]:
# Installazione delle dipendenze (e clone del repo su Google Colab)
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/Corso-Computer-Vision'):
        !git clone https://github.com/SalvoSamba01/Corso-Computer-Vision
    %cd /content/Corso-Computer-Vision

!pip install -q "numpy>=2.0"
!pip install -q basicsr facexlib gfpgan
!pip install -q realesrgan
!pip install -q ultralytics huggingface_hub
!pip install -q opencv-python-headless matplotlib tqdm pandas

**Import librerie Giorno 3.** Alle librerie standard si aggiungono `hashlib`, `json`, `datetime` per il sistema di logging (sezione 3). Da `cv_utils`: `psnr` e `degrada_immagine`.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import hashlib
import json
import datetime
import warnings
warnings.filterwarnings('ignore')

import sys, os
if os.path.exists('/content/Corso-Computer-Vision'):
    sys.path.append('/content/Corso-Computer-Vision/utils')
else:
    sys.path.append('../utils')
from cv_utils import (mostra_frame, mostra_confronto, mostra_griglia,
                       stampa_info_video, estrai_frame, degrada_immagine, psnr,
                       converti_video_h264, mostra_video_colab)

In [ ]:
# Configurazione percorsi
import os
from pathlib import Path

if os.path.exists('/content/Corso-Computer-Vision'):
    DATA_DIR  = Path('/content/Corso-Computer-Vision/data/day_3')
    BASE_DIR  = Path('/content/Corso-Computer-Vision')
else:
    DATA_DIR  = Path('../data/day_3')
    BASE_DIR  = Path('..')

ACTION_DIR = DATA_DIR / 'CCVT'    # CCTV Action Recognition
TARGHE_DIR = DATA_DIR / 'Targhe'  # immagini targhe (GT = nome file)

VIDEOS_DIR = BASE_DIR / 'results' / 'videos'
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# Categorie azione CCTV
action_cats = sorted([d for d in ACTION_DIR.iterdir() if d.is_dir()])

# Immagini targhe (png o jpg, GT = stem del filename)
image_files = sorted(list(TARGHE_DIR.glob('*.png')) + list(TARGHE_DIR.glob('*.jpg')))

print(f'CCTV Action Recognition — {len(action_cats)} categorie:')
for cat in action_cats:
    n = len(list(cat.glob('*.mp4')))
    print(f'  {cat.name}: {n} video')

print(f'\nImmagini targhe: {len(image_files)}')
for f in image_files:
    print(f'  {f.name}  →  GT: {f.stem}')
print(f'\nOutput video → {VIDEOS_DIR}')


In [ ]:
# Anteprima del dataset CCTV Action Recognition: primo frame per ogni categoria di azione.
# Mostra la varietà di scene presenti (fall, run, stand, struggle, ecc.).
# ── Preview: un frame per ogni categoria di azione ────────────────────────────
frames_action = []
titoli_action = []

for cat_dir in action_cats:
    vids = sorted(cat_dir.glob('*.mp4'))
    if vids:
        frame = estrai_frame(str(vids[0]), n=1)
        if frame:
            frames_action.append(frame[0])
            titoli_action.append(cat_dir.name)

if frames_action:
    mostra_griglia(frames_action, titoli_action, colonne=2)
    print(f'{len(frames_action)} categorie — clip reali da telecamere di sorveglianza')


---
## 1. Real-ESRGAN: Super-Resolution e Restoration

**Real-ESRGAN** è un modello di image restoration basato su GAN (Generative Adversarial Network), addestrato su degradazioni *sintetiche ma realistiche*.

A differenza dei modelli classici di super-resolution, Real-ESRGAN:
- Gestisce contemporaneamente: **super-resolution**, **denoising**, **deblur**, **de-compression**
- È addestrato su immagini CCTV, video compressi, immagini a bassa risoluzione
- Produce output fino a **4x** la risoluzione originale

**Varianti disponibili:**
- `RealESRGAN_x4plus`: uso generale, 4x upscale
- `RealESRGAN_x2plus`: 2x upscale, più veloce
- `RealESRGAN_x4plus_anime_6B`: ottimizzato per contenuti anime/cartoon
- `realesr-general-x4v3`: modello più leggero per degradazioni generali

Peso dei modelli: **~65MB** (scaricati automaticamente dal repo ufficiale)

In [ ]:
# Patch di compatibilità: basicsr importa torchvision.transforms.functional_tensor,
# rimosso in torchvision >= 0.15. Creiamo un modulo stub per evitare l'ImportError.
# Necessario eseguire questa cella prima di importare RealESRGANer.
# Fix compatibilità basicsr con torchvision >= 0.15
# In torchvision >= 0.15, functional_tensor è stato rimosso/accorpato in functional
import sys, types
import torchvision.transforms.functional as _F
_ft = types.ModuleType("torchvision.transforms.functional_tensor")
_ft.rgb_to_grayscale = _F.rgb_to_grayscale
sys.modules["torchvision.transforms.functional_tensor"] = _ft
print("✅ Patch torchvision.transforms.functional_tensor applicata")


In [ ]:
import torch
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan.archs.srvgg_arch import SRVGGNetCompact

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# ── Modello 1: RealESRGAN x4 (uso generale, alta qualità) ────────────────────
model_arch = RRDBNet(
    num_in_ch=3, num_out_ch=3, num_feat=64,
    num_block=23, num_grow_ch=32, scale=4
)

upsampler_x4 = RealESRGANer(
    scale=4,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    dni_weight=None,
    model=model_arch,
    tile=0,           # 0 = elaborazione senza tiling (più veloce, richiede più VRAM)
    tile_pad=10,
    pre_pad=0,
    half=True if device=='cuda' else False,  # fp16 su GPU
    device=device
)

print('✅ Real-ESRGAN x4 caricato')

In [ ]:
# ── Modello 2: RealESRGAN x2 (più veloce) ────────────────────────────────────
model_arch_x2 = RRDBNet(
    num_in_ch=3, num_out_ch=3, num_feat=64,
    num_block=23, num_grow_ch=32, scale=2
)

upsampler_x2 = RealESRGANer(
    scale=2,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
    model=model_arch_x2,
    tile=0,
    tile_pad=10,
    pre_pad=0,
    half=True if device=='cuda' else False,
    device=device
)

print('✅ Real-ESRGAN x2 caricato')

**Funzione wrapper Real-ESRGAN.** `esegui_realesrgan()` gestisce la conversione `BGR→RGB` (Real-ESRGAN lavora in RGB), chiama `upsampler.enhance()` e riconverte l'output in BGR. È la funzione centrale usata in tutte le demo di restoration.

In [ ]:
def esegui_realesrgan(img_bgr: np.ndarray, upsampler,
                       outscale: float = None) -> np.ndarray:
    """
    Applica Real-ESRGAN a un'immagine BGR.
    Restituisce l'immagine restored in BGR.
    """
    # Real-ESRGAN lavora in RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    output, _ = upsampler.enhance(img_rgb, outscale=outscale)
    return cv2.cvtColor(output, cv2.COLOR_RGB2BGR)

---
## 2. Restoration di frame CCTV

In [ ]:
# Selezioniamo la categoria 'fall' (caduta) — anomalia tipica nei sistemi di sorveglianza.
# Se il video non esiste, creiamo un frame sintetico degradato per la demo.
# Selezioniamo un video di sorveglianza con azione anomala
# Il dataset CCTV Action Recognition contiene clip reali da telecamere di sicurezza.
# Usiamo la categoria "fall" (caduta): tipica anomalia nei sistemi di sorveglianza.

action_fall = sorted((ACTION_DIR / 'fall').glob('*.mp4'))
VIDEO_CCTV  = str(action_fall[0]) if action_fall else None

if VIDEO_CCTV:
    print(f'Video selezionato: fall/{Path(VIDEO_CCTV).name}')
    stampa_info_video(VIDEO_CCTV)
    frame_cctv = estrai_frame(VIDEO_CCTV, n=1)[0]
    mostra_frame(frame_cctv, 'Frame CCTV — categoria: fall')
else:
    print('Nessun video trovato — creazione frame sintetico per demo')
    frame_hd   = np.random.randint(50, 200, (480, 640, 3), dtype=np.uint8)
    frame_cctv = degrada_immagine(frame_hd, scala=0.25, blur=3, rumore=20)


**Super-resolution ×4 su frame CCTV.** Applichiamo `RealESRGAN_x4plus` al frame selezionato. Il modello aumenta la risoluzione di 4× migliorando contemporaneamente nitidezza, deblur e riduzione del rumore.

In [ ]:
# ── 2.1 Super-resolution x4 su frame CCTV ────────────────────────────────────
print('Elaborazione Real-ESRGAN x4... (può richiedere qualche secondo su CPU)')

frame_restored_x4 = esegui_realesrgan(frame_cctv, upsampler_x4)

print(f'Originale:  {frame_cctv.shape[1]}x{frame_cctv.shape[0]} px')
print(f'Restored x4: {frame_restored_x4.shape[1]}x{frame_restored_x4.shape[0]} px')

# Confronto
mostra_confronto(frame_cctv, frame_restored_x4,
                 f'CCTV originale ({frame_cctv.shape[1]}x{frame_cctv.shape[0]})',
                 f'Real-ESRGAN x4 ({frame_restored_x4.shape[1]}x{frame_restored_x4.shape[0]})')

**Confronto ×2 vs ×4.** Confrontiamo le due varianti sullo stesso frame. Per un confronto visivo equo, riportiamo tutte le immagini alla stessa risoluzione con resize bicubico (nearest per l'originale).

In [ ]:
# ── 2.2 Confronto x2 vs x4 ───────────────────────────────────────────────────
frame_restored_x2 = esegui_realesrgan(frame_cctv, upsampler_x2)

# Portiamo tutto alla stessa risoluzione per confronto visivo
H_x4, W_x4 = frame_restored_x4.shape[:2]
frame_orig_up = cv2.resize(frame_cctv, (W_x4, H_x4), interpolation=cv2.INTER_NEAREST)
frame_x2_up   = cv2.resize(frame_restored_x2, (W_x4, H_x4), interpolation=cv2.INTER_LANCZOS4)

mostra_griglia(
    [frame_orig_up, frame_x2_up, frame_restored_x4],
    ['Originale (upsampled bicubico)', 'Real-ESRGAN x2', 'Real-ESRGAN x4'],
    colonne=2
)

### 2.4 SSIM — Structural Similarity Index

Il **PSNR** misura l'errore quadratico medio, ma non sempre riflette la percezione umana.  
Lo **SSIM** valuta tre componenti locali:
- **Luminanza** — livello medio di grigio
- **Contrasto** — deviazione standard locale
- **Struttura** — correlazione dei pattern (bordi, texture)

**Intervalli di qualità:** SSIM > 0.9 → ottima | 0.7–0.9 → buona | < 0.7 → degradata

**Vantaggio rispetto al PSNR:** due immagini con lo stesso PSNR ma diversa struttura (es. blur diffuso vs rumore casuale) hanno SSIM molto diversi. La mappa SSIM mostra esattamente *dove* si concentrano le differenze strutturali.

### 2.6 Confronto restoration su diverse categorie di azione

Il dataset **CCTV Action Recognition** contiene clip da telecamere reali con scene eterogenee: cadute, risse, appostamenti, fughe, ecc. Scene diverse → caratteristiche visive diverse (illuminazione, movimento, densità di dettaglio) → comportamento diverso della restoration.

Confrontiamo PSNR e SSIM dopo Real-ESRGAN su alcune categorie contrastanti per capire dove la restoration porta i maggiori benefici.

**Degradazione sintetica per misurare il PSNR.** Per calcolare il PSNR abbiamo bisogno del ground truth. Degradiamo artificialmente un frame ad alta risoluzione: così conosciamo la verità e possiamo misurare il miglioramento oggettivo prima e dopo la restoration.

In [ ]:
# ── 2.3 Restoration su immagine artificialmente degradata (PSNR misurabile) ──
# Per misurare il PSNR abbiamo bisogno dell'immagine originale (ground truth)
# Usiamo un frame HD e lo degradiamo noi stessi

# Prendiamo un frame dal video ad alta risoluzione se disponibile
# altrimenti usiamo il frame CCTV ridimensionato come proxy
frame_gt = cv2.resize(frame_cctv, 
                       (frame_cctv.shape[1]*4, frame_cctv.shape[0]*4),
                       interpolation=cv2.INTER_LANCZOS4)
frame_degradato = degrada_immagine(frame_gt, scala=0.25, blur=5, rumore=15)

print('Applicazione Real-ESRGAN sul frame degradato sinteticamente...')
frame_restored_sintetico = esegui_realesrgan(frame_degradato, upsampler_x4)

# Riportiamo tutte le immagini alla stessa risoluzione per il confronto
H_gt, W_gt = frame_gt.shape[:2]
frame_deg_up  = cv2.resize(frame_degradato, (W_gt, H_gt), interpolation=cv2.INTER_NEAREST)
frame_rest_up = cv2.resize(frame_restored_sintetico, (W_gt, H_gt), interpolation=cv2.INTER_LANCZOS4)

psnr_originale = psnr(frame_gt, frame_deg_up)
psnr_restored  = psnr(frame_gt, frame_rest_up)

print(f'\nPSNR originale vs degradato: {psnr_originale:.2f} dB')
print(f'PSNR originale vs restored:  {psnr_restored:.2f} dB')
print(f'Miglioramento PSNR:          {psnr_restored - psnr_originale:+.2f} dB')

mostra_griglia(
    [frame_gt, frame_deg_up, frame_rest_up],
    [f'Ground truth', f'Degradato (PSNR={psnr_originale:.1f}dB)', f'Restored (PSNR={psnr_restored:.1f}dB)'],
    colonne=3
)

In [ ]:
# ── 2.4 SSIM — Structural Similarity Index ───────────────────────────────────
try:
    from skimage.metrics import structural_similarity as ssim_fn
except ImportError:
    import subprocess; subprocess.run(['pip', 'install', '-q', 'scikit-image'])
    from skimage.metrics import structural_similarity as ssim_fn

def calcola_ssim(img_ref: np.ndarray, img_test: np.ndarray) -> float:
    """Calcola SSIM in scala di grigi; ridimensiona img_test se necessario."""
    if img_ref.shape != img_test.shape:
        img_test = cv2.resize(img_test, (img_ref.shape[1], img_ref.shape[0]),
                              interpolation=cv2.INTER_LANCZOS4)
    ref_g = cv2.cvtColor(img_ref,  cv2.COLOR_BGR2GRAY)
    tst_g = cv2.cvtColor(img_test, cv2.COLOR_BGR2GRAY)
    return ssim_fn(ref_g, tst_g, data_range=255)

# Utilizziamo le immagini già calcolate nella cella precedente:
#   frame_gt       → ground truth alta risoluzione
#   frame_deg_up   → degradato + upsampled alla risoluzione GT
#   frame_rest_up  → restored ESRGAN + upsampled

print(f'{"Scenario":<25} {"PSNR (dB)":>12} {"SSIM":>8}')
print('-' * 48)
for nome, img in [('Degradato', frame_deg_up), ('Restored (ESRGAN)', frame_rest_up)]:
    p = psnr(frame_gt, img)
    s = calcola_ssim(frame_gt, img)
    print(f'{nome:<25} {p:>12.2f} {s:>8.4f}')
print('\nℹ️  SSIM > 0.9 → ottima  |  0.7–0.9 → buona  |  < 0.7 → degradata')

# Mappa SSIM: dove si concentrano le differenze strutturali?
ref_g  = cv2.cvtColor(frame_gt,      cv2.COLOR_BGR2GRAY)
deg_g  = cv2.cvtColor(frame_deg_up,  cv2.COLOR_BGR2GRAY)
rest_g = cv2.cvtColor(frame_rest_up, cv2.COLOR_BGR2GRAY)

_, ssim_map_deg  = ssim_fn(ref_g, deg_g,  data_range=255, full=True)
_, ssim_map_rest = ssim_fn(ref_g, rest_g, data_range=255, full=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(ref_g, cmap='gray')
axes[0].set_title('Ground truth'); axes[0].axis('off')
axes[1].imshow(ssim_map_deg, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_title(f'Mappa SSIM — Degradato\nSSIM={calcola_ssim(frame_gt, frame_deg_up):.4f}')
axes[1].axis('off')
axes[2].imshow(ssim_map_rest, cmap='RdYlGn', vmin=0, vmax=1)
axes[2].set_title(f'Mappa SSIM — Restored\nSSIM={calcola_ssim(frame_gt, frame_rest_up):.4f}')
axes[2].axis('off')
plt.suptitle('Mappa SSIM: verde = alta similarità, rosso = bassa similarità', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.6 Confronto restoration: diverse categorie di azione ────────────────────
# Scegliamo categorie visivamente contrastanti
CAT_TEST = ['fall', 'run', 'stand', 'struggle']

risultati_categorie = []

for cat_name in CAT_TEST:
    cat_dir = ACTION_DIR / cat_name
    vids = sorted(cat_dir.glob('*.mp4'))
    if not vids:
        continue
    f = estrai_frame(str(vids[0]), n=1)
    if not f:
        continue
    frame = f[0]

    # Degradazione sintetica + restoration x2
    frame_deg  = degrada_immagine(frame, scala=0.5, blur=5, rumore=15)
    frame_rest = esegui_realesrgan(frame_deg, upsampler_x2)

    H_r, W_r    = frame_rest.shape[:2]
    frame_gt_c  = cv2.resize(frame,     (W_r, H_r), interpolation=cv2.INTER_LANCZOS4)
    frame_deg_c = cv2.resize(frame_deg, (W_r, H_r), interpolation=cv2.INTER_NEAREST)

    p_deg  = psnr(frame_gt_c, frame_deg_c)
    p_rest = psnr(frame_gt_c, frame_rest)
    s_deg  = calcola_ssim(frame_gt_c, frame_deg_c)
    s_rest = calcola_ssim(frame_gt_c, frame_rest)

    print(f'\n  {cat_name}:')
    print(f'    Degradato → PSNR={p_deg:.1f}dB, SSIM={s_deg:.4f}')
    print(f'    Restored  → PSNR={p_rest:.1f}dB, SSIM={s_rest:.4f}  (Δ={p_rest-p_deg:+.1f}dB)')
    mostra_confronto(frame_deg_c, frame_rest,
                     f'{cat_name} — degradato (PSNR={p_deg:.1f}dB)',
                     f'{cat_name} — restored  (PSNR={p_rest:.1f}dB)')

    risultati_categorie.append({
        'categoria': cat_name,
        'psnr_deg':  p_deg,  'psnr_rest': p_rest,
        'ssim_deg':  s_deg,  'ssim_rest': s_rest,
    })

# Barplot interattivo comparativo PSNR e SSIM
if risultati_categorie:
    import pandas as pd
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    df_cat = pd.DataFrame(risultati_categorie)
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('PSNR per categoria', 'SSIM per categoria'))

    for col_deg, col_rest, col_idx, ylabel in [
        ('psnr_deg', 'psnr_rest', 1, 'PSNR (dB)'),
        ('ssim_deg', 'ssim_rest', 2, 'SSIM'),
    ]:
        fig.add_trace(go.Bar(
            x=df_cat['categoria'], y=df_cat[col_deg],
            name='Degradato', marker_color='coral', opacity=0.85,
            showlegend=(col_idx == 1)
        ), row=1, col=col_idx)
        fig.add_trace(go.Bar(
            x=df_cat['categoria'], y=df_cat[col_rest],
            name='Restored', marker_color='steelblue', opacity=0.85,
            showlegend=(col_idx == 1)
        ), row=1, col=col_idx)

    fig.update_layout(
        height=400, barmode='group',
        title='Qualità restoration per tipo di scena CCTV',
        margin=dict(l=10, r=10, t=70, b=40)
    )
    fig.update_yaxes(title_text='PSNR (dB)', row=1, col=1)
    fig.update_yaxes(title_text='SSIM', row=1, col=2)
    fig.show()

In [ ]:
# ── 2.5 Restoration su sequenza video ────────────────────────────────────────
# Processiamo i primi N frame del video CCTV
N_FRAME_REST = 30  # pochi frame: Real-ESRGAN è lento su CPU
OUTPUT_REST     = str(VIDEOS_DIR / 'day3_cctv_restored.mp4')
OUTPUT_REST_TMP = str(VIDEOS_DIR / 'day3_cctv_restored_tmp.mp4')

if VIDEO_CCTV:
    cap = cv2.VideoCapture(VIDEO_CCTV)
    fps_r = cap.get(cv2.CAP_PROP_FPS)
    w_in  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_in  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Il video di output ha risoluzione 4x
    writer = cv2.VideoWriter(OUTPUT_REST_TMP, cv2.VideoWriter_fourcc(*'mp4v'),
                             fps_r, (w_in * 4, h_in * 4))

    for i in tqdm(range(N_FRAME_REST), desc='Video restoration'):
        ret, frame = cap.read()
        if not ret:
            break
        restored = esegui_realesrgan(frame, upsampler_x4)
        writer.write(restored)

    cap.release()
    writer.release()

    OUTPUT_REST = mostra_video_colab(OUTPUT_REST_TMP, OUTPUT_REST)
    print(f'\nVideo restored salvato: {OUTPUT_REST}')
else:
    print('Nessun video disponibile — saltiamo la sezione')

---
## 3. Face Restoration con GFPGAN

**GFPGAN** (Generative Facial Prior GAN — Tencent ARC) è un modello specializzato nel ripristino di **volti degradati** in fotografie e video di sorveglianza.

A differenza di Real-ESRGAN (uso generale), GFPGAN:
- È ottimizzato specificamente per la struttura del volto umano
- Sfrutta un **prior facciale** (GAN pre-addestrata su volti ad alta risoluzione) per allucinare dettagli realistici
- Include **face detection e alignment automatici**: basta passare il frame, il modello trova e restora i volti da solo
- Supporta sia volti frontali che parzialmente ruotati/occlusi

**Applicazioni tipiche in sorveglianza:**
- Identificazione di soggetti da telecamere a bassa risoluzione
- Pre-processing per sistemi di face recognition
- Analisi forense di immagini degradate

> `gfpgan` è già installata come dipendenza di `realesrgan` — nessun `pip install` aggiuntivo.

In [ ]:
from gfpgan import GFPGANer

# GFPGANv1.3: bilanciamento qualità/naturalezza (consigliato)
# GFPGANv1.4: più nitido, talvolta eccessivamente 'sintetico'
restorer_gfpgan = GFPGANer(
    model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
    upscale=2,
    arch='clean',
    channel_multiplier=2,
    bg_upsampler=None,  # sfondo non modificato
    device=device,
)
print('✅ GFPGAN v1.3 caricato')

In [ ]:
# esegui_gfpgan(): wrapper attorno a GFPGANer.enhance() — rileva e restora i volti automaticamente.
# ID_PERSONA (0-9): variabile da modificare per confrontare persone diverse del dataset SCface.
def esegui_gfpgan(img_bgr: np.ndarray,
                   only_center_face: bool = False) -> np.ndarray:
    """
    Applica GFPGAN all'immagine BGR.
    Rileva e restora automaticamente tutti i volti presenti.
    Restituisce l'immagine con i volti restaurati (upscalata x2).
    """
    _, _, output = restorer_gfpgan.enhance(
        img_bgr,
        has_aligned=False,
        only_center_face=only_center_face,
        paste_back=True,
    )
    return output


# ── Dataset SCface: cartelle persona ──────────────────────────────────────────
SCFACE_FACES_DIR = DATA_DIR / 'CCVT_FACES'
persona_dirs = sorted(SCFACE_FACES_DIR.glob('persona_*'))

# ┌─────────────────────────────────────────────────────────┐
# │  CAMBIA QUI per selezionare una persona diversa (0–9)  │
ID_PERSONA = 0
# └─────────────────────────────────────────────────────────┘

PERSONA = persona_dirs[ID_PERSONA]

print(f'{len(persona_dirs)} persone disponibili:')
for i, d in enumerate(persona_dirs):
    imgs  = list(d.glob('*.jpg')) + list(d.glob('*.JPG'))
    n_mug  = len([f for f in imgs if 'mugshot' in f.name])
    n_surv = len([f for f in imgs if 'surveillance' in f.name])
    marker = '  ◀ selezionata' if i == ID_PERSONA else ''
    print(f'  [{i}] {d.name}: {n_mug} mugshot, {n_surv} surveillance{marker}')


**Mugshot — ground truth visivo.** Carichiamo le due foto di riferimento della persona selezionata (originale + cropped). Sono le immagini di qualità massima disponibili: il benchmark visivo per valutare quanto la restoration si avvicina alla realtà.

In [ ]:
# ── Mugshot: ground truth alta qualità ───────────────────────────────────────
mugshot_orig = cv2.imread(str(next(PERSONA.glob('mugshot_original_*.jpg'))))
mugshot_crop = cv2.imread(str(next(PERSONA.glob('mugshot_cropped_*.JPG'))))

print(f'Mugshot originale: {mugshot_orig.shape[1]}x{mugshot_orig.shape[0]} px')
print(f'Mugshot cropped:   {mugshot_crop.shape[1]}x{mugshot_crop.shape[0]} px')

mostra_confronto(
    mugshot_orig,
    mugshot_crop,
    f'Mugshot originale ({mugshot_orig.shape[1]}x{mugshot_orig.shape[0]})',
    f'Mugshot cropped   ({mugshot_crop.shape[1]}x{mugshot_crop.shape[0]})',
)


In [ ]:
# ── Restoration GFPGAN su tutte le immagini di sorveglianza ─────────────────
# Naming: surveillance_{ID}_cam{N}_{D}.jpg  (N=camera 1-7, D=distanza 1-3)
surv_files = sorted(PERSONA.glob('surveillance_*.jpg'))
print(f'{len(surv_files)} immagini sorveglianza trovate — applicazione GFPGAN...')

frames_grid = []
titoli_grid = []

for f in tqdm(surv_files, desc='Face restoration'):
    parts = f.stem.split('_')   # ['surveillance', '007', 'cam1', '1']
    label = f'{parts[-2]} dist.{parts[-1]}'  # 'cam1 dist.1'

    img          = cv2.imread(str(f))
    img_restored = esegui_gfpgan(img)

    # Upscale bicubico dell'originale per confronto visivo alla stessa risoluzione
    H_r, W_r = img_restored.shape[:2]
    img_up = cv2.resize(img, (W_r, H_r), interpolation=cv2.INTER_NEAREST)

    frames_grid.append(img_up)
    frames_grid.append(img_restored)
    titoli_grid.append(f'Originale — {label}')
    titoli_grid.append(f'GFPGAN — {label}')

# Griglia: colonne=4 → ogni riga = 2 coppie (originale | restored)
mostra_griglia(frames_grid, titoli_grid, colonne=4)

### Osservazioni
- Le immagini di sorveglianza mostrano degradazione **crescente con la distanza** (`dist.1` = più vicino, `dist.3` = più lontano)
- GFPGAN recupera dettagli facciali anche da immagini molto basse risoluzione, allucinando texture realistiche
- Confronta i risultati con il **mugshot**: quanto si avvicina la restoration all'immagine di riferimento?
- Prova a cambiare persona modificando `ID_PERSONA = X` (X da 0 a 9) nella cella precedente


---
## 4. Ripetibilità: logging, parametri e hash

In un contesto professionale, la **ripetibilità** è fondamentale:
- Devo poter ricostruire esattamente come è stata prodotta un'immagine/output
- Devo poter verificare che un'immagine non sia stata modificata dopo il processing
- Devo tracciare quale versione del modello ha prodotto quale output

**Strumenti che usiamo:**
- **Hash SHA-256**: impronta digitale unica dell'immagine
- **JSON log**: registro di ogni operazione eseguita
- **Parametri salvati**: configurazione completa riproducibile

In [ ]:
# SessionLogger: classe per tracciare ogni operazione con timestamp, hash I/O e parametri.
# hash_immagine(): SHA-256 dei pixel — garantisce l'integrità e la tracciabilità dell'output.
def hash_immagine(img: np.ndarray) -> str:
    """Calcola l'hash SHA-256 dei pixel dell'immagine."""
    return hashlib.sha256(img.tobytes()).hexdigest()


class SessionLogger:
    """
    Logger per tracciare operazioni di processing su immagini/video.
    Salva un file JSON con tutti i dettagli della sessione.
    """

    def __init__(self, output_path: str):
        self.output_path = output_path
        self.log = {
            'sessione': {
                'data': datetime.datetime.now().isoformat(),
                'device': device,
            },
            'operazioni': []
        }

    def registra(self, operazione: str, input_info: dict,
                  output_info: dict, parametri: dict):
        """Registra un'operazione nel log."""
        entry = {
            'timestamp': datetime.datetime.now().isoformat(),
            'operazione': operazione,
            'input': input_info,
            'output': output_info,
            'parametri': parametri,
        }
        self.log['operazioni'].append(entry)
        self._salva()

    def _salva(self):
        with open(self.output_path, 'w') as f:
            json.dump(self.log, f, indent=2, ensure_ascii=False)

    def stampa_riepilogo(self):
        print(f'Log sessione: {self.output_path}')
        print(f'Operazioni registrate: {len(self.log["operazioni"])}')
        for op in self.log['operazioni']:
            print(f"  [{op['timestamp'][:19]}] {op['operazione']}")


In [ ]:
# ── 3.1 Esempio di pipeline con logging ──────────────────────────────────────
LOG_PATH = '/tmp/sessione_restoration.json'
logger = SessionLogger(LOG_PATH)

# Parametri della sessione
PARAMS_RESTORATION = {
    'modello': 'RealESRGAN_x4plus',
    'scale': 4,
    'tile': 0,
    'half_precision': device == 'cuda',
    'device': device,
}

# --- Operazione 1: restoration di un frame ---
frame_input = frame_cctv.copy()
hash_input = hash_immagine(frame_input)

frame_output = esegui_realesrgan(frame_input, upsampler_x4)
hash_output = hash_immagine(frame_output)

logger.registra(
    operazione='real_esrgan_restoration',
    input_info={
        'risoluzione': f'{frame_input.shape[1]}x{frame_input.shape[0]}',
        'sha256': hash_input,
    },
    output_info={
        'risoluzione': f'{frame_output.shape[1]}x{frame_output.shape[0]}',
        'sha256': hash_output,
    },
    parametri=PARAMS_RESTORATION
)

print(f'Hash input:  {hash_input[:16]}...')
print(f'Hash output: {hash_output[:16]}...')
print('Gli hash sono diversi ✅ (l\'immagine è stata modificata, come atteso)')

In [ ]:
# ── 3.2 Verifica integrità: stessa immagine → stesso hash ────────────────────
frame_copia = frame_output.copy()
hash_copia = hash_immagine(frame_copia)

print(f'Hash output originale: {hash_output[:32]}...')
print(f'Hash copia:            {hash_copia[:32]}...')
print(f'Identici: {hash_output == hash_copia} ✅')

# Modifica minima → hash completamente diverso
frame_modificato = frame_output.copy()
frame_modificato[0, 0, 0] += 1  # cambia un solo pixel
hash_modificato = hash_immagine(frame_modificato)

print(f'\nHash dopo modifica 1px: {hash_modificato[:32]}...')
print(f'Diverso dall\'originale: {hash_output != hash_modificato} ✅')

In [ ]:
# ── 3.3 Salvataggio e caricamento parametri ───────────────────────────────────
PARAMS_FILE = '/tmp/params_restoration.json'

# Salvataggio
with open(PARAMS_FILE, 'w') as f:
    json.dump(PARAMS_RESTORATION, f, indent=2)
print(f'Parametri salvati in: {PARAMS_FILE}')

# Caricamento e riuso
with open(PARAMS_FILE, 'r') as f:
    params_caricati = json.load(f)

print('\nParametri caricati:')
for k, v in params_caricati.items():
    print(f'  {k}: {v}')

logger.stampa_riepilogo()

---
## 5. Caso Studio: License Plate Restoration

Mettiamo insieme tutti gli strumenti visti oggi in una **pipeline end-to-end**
per la restoration visiva di targhe degradate:

```
Immagine targa  →  YOLO detection  →  crop  →  Real-ESRGAN restoration
```

Il **ground truth** è codificato nel nome del file: confronteremo visivamente
il crop originale e quello restaurato, con la targa GT come titolo.


### Ground Truth nel nome file

Nel dataset scelto il **ground truth di ogni targa è codificato direttamente nel nome del file**.

Esempio:
```
TN19A5532.png  →  targa GT: TN19A5532
WB04B0829.png  →  targa GT: WB04B0829
```

Visualizzeremo ogni coppia (originale | restored) con il testo GT come titolo.


In [ ]:
# ── Visualizzazione dataset targhe con GT da filename ───────────────────────
print(f'{len(image_files)} immagini disponibili:\n')

frames_gt = []
titoli_gt = []

for img_path in image_files:
    targa_gt_label = img_path.stem  # GT = nome file senza estensione
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    print(f'  {img_path.name:25s}  GT: {targa_gt_label}  ({img.shape[1]}x{img.shape[0]})')
    frames_gt.append(img)
    titoli_gt.append(f'GT: {targa_gt_label}')

mostra_griglia(frames_gt, titoli_gt, colonne=3)


### 4.1 Caricamento modelli

**YOLOv11** pre-addestrato per il riconoscimento di targhe
(fonte: [morsetechlab/yolov11-license-plate-detection](https://huggingface.co/morsetechlab/yolov11-license-plate-detection))


In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Download dei pesi da HuggingFace (~19MB, variante small)
model_path = hf_hub_download(
    repo_id='morsetechlab/yolov11-license-plate-detection',
    filename='license-plate-finetune-v1s.pt'
)

model_plate = YOLO(model_path)
print(f'Modello targhe caricato: {model_path}')
print(f'Classi: {model_plate.names}')

### 4.2 Restoration delle targhe

Per ogni immagine del dataset:
1. **YOLO** rileva la targa e ne estrae il crop
2. **Real-ESRGAN ×4** restora il crop
3. Visualizziamo l'originale e la versione restaurata affiancate, con il testo GT come titolo


In [ ]:
def rileva_targhe(img_bgr: np.ndarray, conf_th: float = 0.3) -> list:
    """
    Rileva le targhe nell'immagine.
    Restituisce lista di (x1, y1, x2, y2, conf).
    """
    ris = model_plate(img_bgr, verbose=False, conf=conf_th)
    boxes = []
    for box in ris[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        boxes.append((x1, y1, x2, y2, conf))
    return boxes


def estrai_crop_targa(img: np.ndarray, x1, y1, x2, y2,
                        padding: int = 5) -> np.ndarray:
    """Estrae il crop della targa con padding."""
    H, W = img.shape[:2]
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(W, x2 + padding)
    y2 = min(H, y2 + padding)
    return img[y1:y2, x1:x2]

### 4.3 Pre-processing classico

Prima di passare il crop a Real-ESRGAN, applichiamo tre operazioni classiche con OpenCV:

| Step | Tecnica | Effetto |
|------|---------|--------|
| 1 | **Bilateral filter** | Riduce il rumore preservando i bordi delle lettere |
| 2 | **CLAHE** (spazio LAB) | Normalizza il contrasto locale senza saturare |
| 3 | **Unsharp masking** | Accentua i bordi, aumenta la leggibilità |

Confronteremo **4 versioni** per ogni targa:
> Originale · Pre-processato · ESRGAN ×4 · Preproc + ESRGAN ×4


In [ ]:
def preprocess_targa(img_bgr: np.ndarray,
                      bilateral: bool = True,
                      clahe: bool = True,
                      unsharp: bool = True) -> np.ndarray:
    """
    Pre-processing classico sul crop della targa.
    Le tre operazioni sono indipendenti e disattivabili singolarmente.
    """
    img = img_bgr.copy()

    # 1. Bilateral filter: denoise edge-preserving
    #    d=5: neighbourhood piccolo (crop già piccolo)
    #    sigmaColor/Space=50: livello di smoothing moderato
    if bilateral:
        img = cv2.bilateralFilter(img, d=5, sigmaColor=50, sigmaSpace=50)

    # 2. CLAHE nello spazio LAB: agisce solo sul canale Luminanza
    #    clipLimit=2.0: limita il contrasto locale per evitare artefatti
    if clahe:
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
        lab[:, :, 0] = clahe_obj.apply(lab[:, :, 0])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    # 3. Unsharp masking: sharpening = originale - versione blurrata
    #    addWeighted(src, 1.5, blur, -0.5) → accentua i dettagli ad alta frequenza
    if unsharp:
        blur = cv2.GaussianBlur(img, (0, 0), sigmaX=2)
        img = cv2.addWeighted(img, 1.5, blur, -0.5, 0)

    return img


print('✅ preprocess_targa definita')


**Pipeline completa su tutte le targhe.** Per ogni immagine: detection YOLO → crop targa → pre-processing classico (bilateral + CLAHE + unsharp) → Real-ESRGAN ×4. Il confronto a 4 colonne per riga mostra il contributo di ogni step.

In [ ]:
# ── Pipeline: detection → crop → [preprocess] → restoration ─────────────────
frames_rest = []
titoli_rest = []

for img_path in tqdm(image_files, desc='License plate restoration'):
    gt  = img_path.stem
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    # Detection
    targhe = rileva_targhe(img, conf_th=0.25)
    if not targhe:
        print(f'  ⚠️  {img_path.name}: nessuna targa rilevata')
        continue

    # Crop con confidenza più alta
    x1, y1, x2, y2, conf = max(targhe, key=lambda t: t[4])
    crop_orig = estrai_crop_targa(img, x1, y1, x2, y2)

    # Pre-processing classico
    crop_pre = preprocess_targa(crop_orig)

    # Restoration: ESRGAN su originale e su pre-processato
    crop_esrgan     = esegui_realesrgan(crop_orig, upsampler_x4)
    crop_pre_esrgan = esegui_realesrgan(crop_pre,  upsampler_x4)

    # Uniforma risoluzione per il confronto visivo
    H_r, W_r = crop_esrgan.shape[:2]
    crop_up     = cv2.resize(crop_orig, (W_r, H_r), interpolation=cv2.INTER_NEAREST)
    crop_pre_up = cv2.resize(crop_pre,  (W_r, H_r), interpolation=cv2.INTER_NEAREST)

    frames_rest.extend([crop_up, crop_pre_up, crop_esrgan, crop_pre_esrgan])
    titoli_rest.extend([
        f'GT: {gt}\nOriginale',
        f'GT: {gt}\nPre-processato',
        f'GT: {gt}\nESRGAN ×4',
        f'GT: {gt}\nPreproc + ESRGAN ×4',
    ])

mostra_griglia(frames_rest, titoli_rest, colonne=4)

---
## 6. Riepilogo del corso

### Giorno 3 — Cosa abbiamo fatto:
1. **Restoration** di frame CCTV con Real-ESRGAN (super-resolution 4x)
2. **Face restoration** con GFPGAN su immagini di sorveglianza (SCface dataset)
3. **Ripetibilità**: logging JSON, hash SHA-256, salvataggio parametri
4. **Caso studio completo**: detection targa → crop → Real-ESRGAN restoration

---

### Riepilogo delle 3 giornate

| | Giorno 1 | Giorno 2 | Giorno 3 |
|---|---|---|---|
| **Topic** | Detection & Tracking | Counting & Traiettorie | Restoration |
| **Librerie** | YOLO, BoxMOT, OpenCV | supervision, norfair | Real-ESRGAN, GFPGAN |
| **Output** | Bounding box + ID | Conteggi + heatmap | Immagini restaurate |
| **Metrica chiave** | Confidenza, ID switch | N° attraversamenti, densità | PSNR, qualità visiva |

### Concetti trasversali applicati:
- **Pre-trained models**: nessun training, solo inference
- **Pipeline composizione**: combinare modelli diversi per task complessi
- **Ripetibilità**: logging, hash, parametri salvati
- **Trade-off**: qualità vs velocità, soglie di confidenza, degrado vs restoration

### Domande finali:
- Quali elementi di questo corso potete applicare subito al vostro lavoro?
- Dove vedereste i maggiori problemi in produzione?
- Come garantireste l'affidabilità di una pipeline come quella del caso studio in un contesto legale/forense?


In [ ]:
# ── ESERCIZIO FINALE ─────────────────────────────────────────────────────────
# Scegliete una delle seguenti sfide:
#
# A) Provate la pipeline di restoration su più immagini dalla cartella day_3
#    e create un report visivo con le coppie originale/restored
#
# B) Combinate il tracking del Giorno 1 con la restoration del Giorno 3:
#    applicare Real-ESRGAN ai crop dei veicoli tracciati in data/day_1
#
# C) Aggiungete la verifica dell'integrità all'output del Giorno 2:
#    salvare le heatmap con hash e log dei parametri

print('Buon lavoro! 🎉')